In [1]:
import pandas as pd
import numpy as np

import warnings 
warnings.simplefilter('ignore')

In [2]:
train = pd.read_csv('/kaggle/input/playground-series-s5e6/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e6/test.csv')
orig = pd.read_csv('/kaggle/input/orig-s5e6/Fertilizer Prediction.csv')

In [3]:
def encode(df):
    soil_type_dict = {'Sandy' : 0, 'Black' : 1, 'Clayey' : 2, 'Red' : 3, 'Loamy' : 4}
    crop_type_dict = {'Paddy' : 0, 'Pulses' : 1, 'Cotton' : 2, 'Tobacco' : 3, 'Wheat' : 4, 'Millets' : 5, 'Barley' : 6, 'Sugarcane' : 7,
                     'Oil seeds' : 8, 'Maize' : 9, 'Ground Nuts' : 10}
    fertilizer_name_dict = {'10-26-26': 0, '14-35-14': 1, '17-17-17': 2, '20-20': 3,
        '28-28': 4, 'DAP': 5, 'Urea': 6}

    df['Soil Type'] = df['Soil Type'].replace(soil_type_dict)
    df['Crop Type'] = df['Crop Type'].replace(crop_type_dict)

    #sadece df verdigimiz icin daha sonrasinda test'e de eklerken sorun olmasin diye
    if 'Fertilizer Name' in df.columns:
        df['Fertilizer Name'] = df['Fertilizer Name'].replace(fertilizer_name_dict)

    df['Soil Type'] = df['Soil Type'].astype('category')
    df['Crop Type'] = df['Crop Type'].astype('category')

    return df


In [4]:
train = encode(train)
test = encode(test)
orig = encode(orig)

train.drop('id', axis=1, inplace=True)
test.drop('id', axis=1, inplace=True)


In [5]:
fertilizer_name_dict = {
    '10-26-26': 0, '14-35-14': 1, '17-17-17': 2, '20-20': 3,
    '28-28': 4, 'DAP': 5, 'Urea': 6
}
inverse_fert_dict = {v: k for k, v in fertilizer_name_dict.items()}

In [ ]:
import os
import time
import gc

import numpy as np
import pandas as pd
from glob import glob
from sklearn.model_selection import KFold
from xgboost import XGBClassifier
import optuna

# -----------------------------------------------------------------------------
# 📥 1) VERİ YÜKLEME (senin kendi dosya yollarına göre düzenle)
# -----------------------------------------------------------------------------
# train = pd.read_csv('train.csv')
# test  = pd.read_csv('test.csv')
# orig  = pd.read_csv('original_data.csv')

# Bir kopya referansı
testt = test.copy()

# -----------------------------------------------------------------------------
# 🎯 2) GLOBAL AYARLAR
# -----------------------------------------------------------------------------
FEATURES    = [c for c in train.columns if c not in ['id', 'Fertilizer Name']]
TARGET_COL  = 'Fertilizer Name'
FOLDS       = 5
N_TRIALS    = 10  # Toplam Optuna deneme sayısı


# Kayıt klasörlerini başta oluştur
os.makedirs("oof_predictions", exist_ok=True)
os.makedirs("submission_predictions", exist_ok=True)

# -----------------------------------------------------------------------------
# 📊 3) MAP@k METRİKLERİ
# -----------------------------------------------------------------------------
def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]
    score = num_hits = 0.0
    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1)
    if not actual:
        return 0.0
    return score / min(len(actual), k)

def mapk(actual, predicted, k=3):
    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

# -----------------------------------------------------------------------------
# 🔬 4) OPTUNA OBJECTIVE FONKSİYONU
#    - Her trial için CV yapıp OOF ve test_preds hesaplar
#    - Her trial sonunda .npy olarak diske yazar
# -----------------------------------------------------------------------------
def objective(trial):
    # 1) Hiperparametre önerileri
    params = {
        'max_depth'            : trial.suggest_int('max_depth', 10, 20),
        'colsample_bytree'     : trial.suggest_float('colsample_bytree', 0.3, 0.85),
        'subsample'            : trial.suggest_float('subsample', 0.3, 1.0),
        'learning_rate'        : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'reg_alpha'            : trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
        'reg_lambda'           : trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
        'min_child_weight'     : trial.suggest_int('min_child_weight', 1, 10),
        'max_delta_step'       : trial.suggest_int('max_delta_step', 0, 10),
        'gamma'                : trial.suggest_float('gamma', 0.0, 1.0),
        'n_estimators'         : trial.suggest_int('n_estimators', 1000, 8000),
        'random_state'         : 42,
        'use_label_encoder'    : False,
        'enable_categorical'   : True,
        'eval_metric'          : 'mlogloss',
        'early_stopping_rounds': 100,
        'tree_method'          : 'gpu_hist',  # CPU için 'hist'
    }
    
    # 2) Model objesi
    model = XGBClassifier(**params, device='cuda')
    
    # 3) Kaç sınıf var?
    n_classes = train[TARGET_COL].nunique()
    
    # 4) OOF ve test_preds dizilerini başlat
    oof = np.zeros((len(train), n_classes))
    test_preds = np.zeros((len(testt), n_classes))
    
    # 5) CV
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)
    fold_scores = []
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
        # 5.1) Train/val ayır
        X_tr, y_tr = train.loc[tr_idx, FEATURES], train.loc[tr_idx, TARGET_COL]
        X_val, y_val = train.loc[val_idx, FEATURES], train.loc[val_idx, TARGET_COL]
        
        # 5.2) Orijinal veri ile augment (isteğe bağlı)
        X_tr = pd.concat([X_tr, orig[FEATURES]], axis=0, ignore_index=True)
        y_tr = pd.concat([y_tr, orig[TARGET_COL]], axis=0, ignore_index=True)
        
        # 5.3) Modeli eğit
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        
        # 5.4) OOF tahmini
        proba_val = model.predict_proba(X_val)
        oof[val_idx] = proba_val
        
        # 5.5) Test tahminini ekle (ortalama)
        test_preds += model.predict_proba(testt[FEATURES]) / FOLDS
        
        # 5.6) Fold MAP@3 skoru
        top3 = np.argsort(proba_val, axis=1)[:, -3:][:, ::-1]
        actual    = [[lab] for lab in y_val]
        predicted = top3.tolist()
        fold_scores.append(mapk(actual, predicted, k=3))
        
        # 5.7) Temizlik
        del X_tr, X_val, y_tr, y_val, proba_val, top3
        gc.collect()
    
    # 6) Kaydet
    num = trial.number
    np.save(f"oof_predictions/oof_trial_{num}.npy", oof)
    #np.save(f"submission_predictions/submission_trial_{num}.npy", test_preds)

    
    sample_sub = pd.read_csv('/kaggle/input/playground-series-s5e6/sample_submission.csv')

    top3_preds = np.argsort(test_preds, axis=1)[:, -3:][:, ::-1]
    top3_str = [
        " ".join(inverse_fert_dict[i] for i in row)
        for row in top3_preds
    ]
    
    submission_df = pd.DataFrame({
        "id": sample_sub["id"],
        "Fertilizer Name": top3_str
    })
    submission_df.to_csv(f"submission_predictions/submission_trial_{num}.csv", index=False)

    
    # 7) Meta veri olarak skoru ekle
    trial.set_user_attr("mean_map3", np.mean(fold_scores))
    
    # 8) Skoru döndür
    return np.mean(fold_scores)

# -----------------------------------------------------------------------------
# 🚀 5) OPTUNA ÇALIŞTIRMASI
# -----------------------------------------------------------------------------
print("=== FAZ 1: Hiperparametre Optimizasyonu Başlatılıyor... ===")
study = optuna.create_study(direction='maximize', study_name='xgb_fertilizer_opt')
study.optimize(objective, n_trials=N_TRIALS)

print("\n=== FAZ 1 Tamamlandı ===")
print(f"En İyi Ortalama MAP@3 Skoru: {study.best_value:.5f}")
print("En İyi Parametreler:", study.best_params)


[I 2025-06-15 12:23:45,748] A new study created in memory with name: xgb_fertilizer_opt


=== FAZ 1: Hiperparametre Optimizasyonu Başlatılıyor... ===


[I 2025-06-15 12:32:02,835] Trial 0 finished with value: 0.3489886666666666 and parameters: {'max_depth': 10, 'colsample_bytree': 0.5816465626014048, 'subsample': 0.9638457773969213, 'learning_rate': 0.04085094473827569, 'reg_alpha': 2.252131186524966, 'reg_lambda': 2.5796349016925078, 'min_child_weight': 1, 'max_delta_step': 1, 'gamma': 0.7305684046796259, 'n_estimators': 2221}. Best is trial 0 with value: 0.3489886666666666.
[I 2025-06-15 13:35:39,915] Trial 1 finished with value: 0.3597422222222222 and parameters: {'max_depth': 12, 'colsample_bytree': 0.5239049365976212, 'subsample': 0.8970539065382792, 'learning_rate': 0.005916892495876141, 'reg_alpha': 0.0010254427780550922, 'reg_lambda': 0.0010548491027967274, 'min_child_weight': 5, 'max_delta_step': 4, 'gamma': 0.5353982434674391, 'n_estimators': 5296}. Best is trial 1 with value: 0.3597422222222222.
[I 2025-06-15 13:56:09,154] Trial 2 finished with value: 0.36506955555555554 and parameters: {'max_depth': 16, 'colsample_bytree':